In [2]:
import re
import sqlite3
import requests
import pandas as pd

headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'accept-language': 'en-US,en;q=0.9',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/144.0.0.0 Safari/537.36',
}

# 32 available markets
MARKETS = [
    'atlanta', 'birmingham', 'charleston', 'charlotte', 'chicago',
    'cincinnati', 'colorado-springs', 'columbus', 'dallas', 'denver',
    'fort-worth', 'ft-myers', 'greenville', 'houston', 'indianapolis',
    'jacksonville', 'kansas-city', 'las-vegas', 'louisville', 'memphis',
    'miami', 'nashville', 'oklahoma-city', 'orlando', 'overland-park',
    'phoenix', 'raleigh-durham', 'san-antonio', 'st-louis', 'tampa',
    'tucson', 'winston-salem',
]

print('Available markets:', ', '.join(MARKETS))

Available markets: atlanta, birmingham, charleston, charlotte, chicago, cincinnati, colorado-springs, columbus, dallas, denver, fort-worth, ft-myers, greenville, houston, indianapolis, jacksonville, kansas-city, las-vegas, louisville, memphis, miami, nashville, oklahoma-city, orlando, overland-park, phoenix, raleigh-durham, san-antonio, st-louis, tampa, tucson, winston-salem


In [4]:
metro_location = input('Enter market (e.g. houston, atlanta, dallas): ').strip().lower()

if metro_location not in MARKETS:
    print(f'Warning: "{metro_location}" not in known markets list')

print(f'Scraping FirstKey Homes: {metro_location}')

Scraping FirstKey Homes: houston


In [5]:
# Fetch the rental page — property data is embedded in the HTML
# The site uses S2A JIT framework with Socket.IO for dynamic loading,
# but the server-side rendered HTML includes property records.
# Note: The SSR may not include all listings (e.g. 48 of 131 for Houston).
# The full dataset requires WebSocket interaction which is not practical here.

url = 'https://www.firstkeyhomes.com/rental-homes/' + metro_location
response = requests.get(url, headers=headers)
response.raise_for_status()

html = response.text
print(f'Page size: {len(html):,} bytes')

# Extract property records from embedded S2A data
# The HTML uses \&q; as escaped quote characters
# Each record starts with: {\&q;_jsonit\&q;:{\&q;_meta\&q;:{\&q;id\&q;:\&q;{ID}\&q;
marker = '{\\&q;_jsonit\\&q;:{\\&q;_meta\\&q;:{\\&q;id\\&q;:\\&q;'
parts = html.split(marker)
print(f'Found {len(parts) - 1} embedded data blocks')

seen_ids = set()
records = []

for part in parts[1:]:
    # Only process blocks that have property data (address field)
    if '\\&q;address\\&q;:\\&q;' not in part:
        continue

    # The _meta.id is the first value right after the split marker
    meta_id = part.split('\\&q;')[0]

    # Deduplicate by _meta.id (same as voyager field, used in URL)
    if meta_id in seen_ids:
        continue
    seen_ids.add(meta_id)

    # Truncate at next record marker for regex safety
    end = part.find('{\\&q;_jsonit\\&q;:{\\&q;_meta\\&q;:{\\&q;id\\&q;:\\&q;', 10)
    if end == -1:
        end = min(len(part), 8000)
    chunk = part[:end]

    def get_str(field):
        m = re.search(r'\\&q;' + field + r'\\&q;:\\&q;([^\\]*)\\&q;', chunk)
        return m.group(1) if m else ''

    def get_num(field):
        m = re.search(r'\\&q;' + field + r'\\&q;:([\d.]+)', chunk)
        return m.group(1) if m else ''

    def get_bool(field):
        m = re.search(r'\\&q;' + field + r'\\&q;:(true|false)', chunk)
        return m.group(1) == 'true' if m else False

    # Extract coordinates from location.coordinates:[lng,lat]
    coords_match = re.search(
        r'\\&q;coordinates\\&q;:\[(-?[\d.]+),(-?[\d.]+)\]', chunk
    )
    lng = coords_match.group(1) if coords_match else ''
    lat = coords_match.group(2) if coords_match else ''

    short_name = get_str('shortName')

    records.append({
        'property_id': meta_id,
        'address': get_str('address'),
        'city': get_str('city'),
        'state': get_str('state'),
        'zip': get_str('zip'),
        'bedrooms': get_num('bedrooms'),
        'bathrooms': get_num('bathrooms'),
        'sqft': get_num('area'),
        'rent': get_num('rent'),
        'available_at': get_str('availableAt'),
        'market': get_str('market'),
        'unit_status': get_num('unitStatus'),
        'self_tour_status': get_num('selfTourStatus'),
        'special_offer': get_bool('specialOffer'),
        'lat': lat,
        'lng': lng,
        'short_name': short_name,
        'link': 'https://www.firstkeyhomes.com/homes-for-rent/'
                + meta_id + '/' + short_name,
        'metro_location': metro_location,
    })

# Check total count from embedded metadata
decoded = html.replace('\\&q;', '"')
count_match = re.search(r'"count":(\d+)', decoded)
total_count = int(count_match.group(1)) if count_match else '?'

print(f'Extracted {len(records)} unique properties (total in market: {total_count})')
if isinstance(total_count, int) and len(records) < total_count:
    print(f'Note: {total_count - len(records)} additional listings load via WebSocket and are not captured here')

Page size: 1,195,641 bytes
Found 2460 embedded data blocks
Extracted 48 unique properties (total in market: 146)
Note: 98 additional listings load via WebSocket and are not captured here


In [10]:
re

48

In [11]:
df = pd.DataFrame(records)

dupes = df.duplicated(subset='property_id', keep='first').sum()
print(f'Duplicate rows: {dupes}')
df = df.drop_duplicates(subset='property_id', keep='first')

df.info()
df.head()

Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   property_id       48 non-null     str  
 1   address           48 non-null     str  
 2   city              48 non-null     str  
 3   state             48 non-null     str  
 4   zip               48 non-null     str  
 5   bedrooms          48 non-null     str  
 6   bathrooms         48 non-null     str  
 7   sqft              48 non-null     str  
 8   rent              48 non-null     str  
 9   available_at      48 non-null     str  
 10  market            48 non-null     str  
 11  unit_status       48 non-null     str  
 12  self_tour_status  48 non-null     str  
 13  special_offer     48 non-null     bool 
 14  lat               48 non-null     str  
 15  lng               48 non-null     str  
 16  short_name        48 non-null     str  
 17  link              48 non-null 

,property_id,address,city,state,zip,bedrooms,bathrooms,sqft,rent,available_at,market,unit_status,self_tour_status,special_offer,lat,lng,short_name,link,metro_location
0,94398701,439 Sky Ridge Drive,Alvin,TX,77511,4,2.5,2295,2580,2026-03-13T12:00:00.000Z,houston,1,1,False,29.4668,-95.2547,439-sky-ridge-drive-alvin-tx-77511,https://www.firstkeyhomes.com/homes-for-rent/9...,houston
1,93218701,321 Lakeway Circle,Anahuac,TX,77514,3,3,2109,2155,2026-02-27T12:00:00.000Z,houston,2,1,False,29.7883,-94.6654,321-lakeway-circle-anahuac-tx-77514,https://www.firstkeyhomes.com/homes-for-rent/9...,houston
2,44358701,4435 Sprangletop Avenue,Baytown,TX,77521,4,2.5,2240,1815,2026-01-14T12:00:00.000Z,houston,3,0,False,29.8184,-94.9928,4435-sprangletop-avenue-baytown-tx-77521,https://www.firstkeyhomes.com/homes-for-rent/4...,houston
3,25588701,2558 Doral Drive,Baytown,TX,77523,4,3,2487,2440,2026-04-08T12:00:00.000Z,houston,1,1,False,29.7099,-94.9572,2558-doral-drive-baytown-tx-77523,https://www.firstkeyhomes.com/homes-for-rent/2...,houston
4,46128701,4612 Ironwood Drive,Baytown,TX,77521,4,2.5,2664,2350,2026-04-20T12:00:00.000Z,houston,1,1,False,29.7681,-94.991,4612-ironwood-drive-baytown-tx-77521,https://www.firstkeyhomes.com/homes-for-rent/4...,houston


In [14]:
df.head(50)

,property_id,address,city,state,zip,bedrooms,bathrooms,sqft,rent,available_at,market,unit_status,self_tour_status,special_offer,lat,lng,short_name,link,metro_location
0,94398701,439 Sky Ridge Drive,Alvin,TX,77511,4,2.5,2295,2580,2026-03-13T12:00:00.000Z,houston,1,1,False,29.4668,-95.2547,439-sky-ridge-drive-alvin-tx-77511,https://www.firstkeyhomes.com/homes-for-rent/9...,houston
1,93218701,321 Lakeway Circle,Anahuac,TX,77514,3,3,2109,2155,2026-02-27T12:00:00.000Z,houston,2,1,False,29.7883,-94.6654,321-lakeway-circle-anahuac-tx-77514,https://www.firstkeyhomes.com/homes-for-rent/9...,houston
2,44358701,4435 Sprangletop Avenue,Baytown,TX,77521,4,2.5,2240,1815,2026-01-14T12:00:00.000Z,houston,3,0,False,29.8184,-94.9928,4435-sprangletop-avenue-baytown-tx-77521,https://www.firstkeyhomes.com/homes-for-rent/4...,houston
3,25588701,2558 Doral Drive,Baytown,TX,77523,4,3,2487,2440,2026-04-08T12:00:00.000Z,houston,1,1,False,29.7099,-94.9572,2558-doral-drive-baytown-tx-77523,https://www.firstkeyhomes.com/homes-for-rent/2...,houston
4,46128701,4612 Ironwood Drive,Baytown,TX,77521,4,2.5,2664,2350,2026-04-20T12:00:00.000Z,houston,1,1,False,29.7681,-94.991,4612-ironwood-drive-baytown-tx-77521,https://www.firstkeyhomes.com/homes-for-rent/4...,houston
5,92158701,215 Brazos Drive,Baytown,TX,77523,5,3,3139,3210,2026-04-22T12:00:00.000Z,houston,1,1,False,29.8243,-94.824,215-brazos-drive-baytown-tx-77523,https://www.firstkeyhomes.com/homes-for-rent/9...,houston
6,70068703,7006 ORCHID St,Baytown,TX,77521,3,2,1620,2025,2026-04-23T12:00:00.000Z,houston,1,1,False,29.7986,-95.0353,7006-orchid-st-baytown-tx-77521-0,https://www.firstkeyhomes.com/homes-for-rent/7...,houston
7,14704602,14701 Country Club Road,Beaumont,TX,77705,3,2,1619,1825,2026-02-11T12:00:00.000Z,houston,3,0,False,29.8698,-94.1407,14701-country-club-road-beaumont-tx-77705,https://www.firstkeyhomes.com/homes-for-rent/1...,houston
8,14728703,14725 Country Club Road,Beaumont,TX,77705,3,2.5,1659,1780,2026-02-24T12:00:00.000Z,houston,3,0,False,29.8698,-94.1402,14725-country-club-road-beaumont-tx-77705,https://www.firstkeyhomes.com/homes-for-rent/1...,houston
9,92458702,245 Road 5102F,Cleveland,TX,77327,3,2,1421,1700,2026-04-02T12:00:00.000Z,houston,1,0,False,30.161,-95.093,245-road-5102f-cleveland-tx-77327,https://www.firstkeyhomes.com/homes-for-rent/9...,houston


In [5]:
df.to_csv('firstkey_homes.csv', index=False)
print(f'Saved {len(df)} rows to firstkey_homes.csv')

Saved 48 rows to firstkey_homes.csv


In [6]:
conn = sqlite3.connect('firstkey_homes.db')
df.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(df)} rows to firstkey_homes.db -> listings table')
conn.close()

Wrote 48 rows to firstkey_homes.db -> listings table
